QLoRa

In [1]:
!pip install -U bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 50.0 MB/s eta 0:00:00


In [17]:
import os
import torch
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig, PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [3]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [4]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
dataset = load_dataset("tatsu-lab/alpaca", split="train[:2000]")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

In [5]:
bnb_config = BitsAndBytesConfig(load_in_4bit=True,
                                bnb_4bit_compute_dtype=torch.float16,
                                bnb_4bit_quant_type="nf4",
                                bnb_4bit_use_double_quant=True)

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [8]:
model.config.use_cache = False

In [9]:
def format_example(ex):
  instruction = ex["instruction"]
  input_text = ex["input"]
  output = ex["output"]

  prompt = instruction if not input_text else f"{instruction}\n\nInput:\n{input_text}"
  return {"text": f"<|user|>\n{prompt}\n<|assistant|>\n{output}{tokenizer.eos_token}"}

In [10]:
dataset = dataset.map(format_example, remove_columns=dataset.column_names)
dataset = dataset.train_test_split(test_size=0.20, seed=42)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

HyperParameters setup

In [20]:
lora_alpha = 32
lora_rank = 16
eval_steps = 10
max_length = 512
lora_dropout = 0.05
learning_rate = 2e-4
num_train_epochs = 1
model_output_dir = "model/qlora_qwen"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lora_target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

In [12]:
lora_config = LoraConfig(r=lora_rank, 
                         lora_alpha=lora_alpha, 
                         lora_dropout=lora_dropout, 
                         bias="none", 
                         task_type="CAUSAL_LM", 
                         target_modules=lora_target_modules)

In [13]:
training_args = SFTConfig(output_dir=model_output_dir,
                          dataset_text_field="text",
                          max_length=max_length,
                          per_device_train_batch_size=4,
                          gradient_accumulation_steps=8,
                          num_train_epochs=num_train_epochs,
                          learning_rate=learning_rate,
                          lr_scheduler_type="cosine",
                          warmup_steps=10,
                          logging_steps=5,
                          save_strategy="epoch", 
                          eval_strategy="epoch",
                          eval_steps=eval_steps,
                          bf16=True,
                          optim="paged_adamw_8bit",
                          report_to="none",
                         )

In [14]:
trainer = SFTTrainer(model=model, args=training_args, train_dataset=dataset["train"], eval_dataset=dataset["test"], peft_config=lora_config)

Adding EOS to train dataset:   0%|          | 0/1600 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1600 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1600 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

In [15]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.336846,1.320512,1.339431,129049.000000,0.661163


TrainOutput(global_step=50, training_loss=1.5044418144226075, metrics={'train_runtime': 1329.3088, 'train_samples_per_second': 1.204, 'train_steps_per_second': 0.038, 'total_flos': 1739959020220416.0, 'train_loss': 1.5044418144226075, 'epoch': 1.0})

In [16]:
trainer.model.print_trainable_parameters()
trainer.save_model(model_output_dir)
tokenizer.save_pretrained(model_output_dir)

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


('model/qlora_qwen/tokenizer_config.json',
 'model/qlora_qwen/chat_template.jinja',
 'model/qlora_qwen/tokenizer.json')

Test Inference

In [18]:
qlora_model = PeftModel.from_pretrained(model, model_output_dir)
model.eval()

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=1536, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=1536, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=1536, out_features=256, bias=True)
            (lora_dropout): ModuleDict(
         

In [23]:
prompt = "<|user|>\nWhat is life?\n<|assistant|>\n"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

In [24]:
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=64, temperature=0.7)
    
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

<|user|>
What is life?
<|assistant|>
Life is a complex and fascinating experience. It involves physical, mental, emotional, and spiritual growth, as well as the ability to create meaningful relationships with others. Life can be both challenging and rewarding, offering opportunities for personal growth and fulfillment.
